In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.constants import (
    ORIGINAL_DATA_PATH,
    PROCESSED_DATA_PATH,
    PREPROCESSING_CATEGORICAL_COLS,
    NO_DATA_SUBJECTS_AFFECTED_ORDER,
)

from src.preprocessing import (
    load_original_dataset,
    filter_to_health_sector,
    filter_to_timeframe,
    set_categorical_dtypes,
    aggregate_to_unique_breaches,
    preprocess_original_dataset,
    validate_processed_dataset
)

In [2]:
df_original = load_original_dataset(ORIGINAL_DATA_PATH)
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206140 entries, 0 to 206139
Data columns (total 11 columns):
 #   Column                      Non-Null Count   Dtype 
---  ------                      --------------   ----- 
 0   BI Reference                206140 non-null  object
 1   Year                        206140 non-null  int64 
 2   Quarter                     206140 non-null  object
 3   Data Subject Type           206140 non-null  object
 4   Data Type                   206140 non-null  object
 5   Decision Taken              206140 non-null  object
 6   Incident Category           206140 non-null  object
 7   Incident Type               206140 non-null  object
 8   No. Data Subjects Affected  206140 non-null  object
 9   Sector                      206140 non-null  object
 10  Time Taken to Report        206140 non-null  object
dtypes: int64(1), object(10)
memory usage: 17.3+ MB


In [3]:
df_health_original = filter_to_health_sector(df_original)
df_health_original.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32127 entries, 12 to 206136
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   bi_reference               32127 non-null  object
 1   year                       32127 non-null  int64 
 2   quarter                    32127 non-null  object
 3   data_subject_type          32127 non-null  object
 4   data_type                  32127 non-null  object
 5   decision_taken             32127 non-null  object
 6   incident_category          32127 non-null  object
 7   incident_type              32127 non-null  object
 8   no_data_subjects_affected  32127 non-null  object
 9   time_taken_to_report       32127 non-null  object
dtypes: int64(1), object(9)
memory usage: 2.7+ MB


In [4]:
df_health_original = filter_to_timeframe(df_health_original)

print(df_health_original[["year", "quarter"]]
    .drop_duplicates()
    .sort_values(by=["year", "quarter"])
    .to_string(index=False))

 year quarter
 2021   Qtr 2
 2021   Qtr 3
 2021   Qtr 4
 2022   Qtr 1
 2022   Qtr 2
 2022   Qtr 3
 2022   Qtr 4
 2023   Qtr 1
 2023   Qtr 2
 2023   Qtr 3
 2023   Qtr 4
 2024   Qtr 1
 2024   Qtr 2
 2024   Qtr 3
 2024   Qtr 4
 2025   Qtr 1
 2025   Qtr 2
 2025   Qtr 3
 2025   Qtr 4


In [5]:
df_health_original = set_categorical_dtypes(df_health_original)
df_health_original.info()
print("\nNo. data subjects affected values (ordered):")
print(df_health_original["no_data_subjects_affected"].cat.categories.tolist())

<class 'pandas.core.frame.DataFrame'>
Index: 22879 entries, 65264 to 206136
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   bi_reference               22879 non-null  category
 1   year                       22879 non-null  category
 2   quarter                    22879 non-null  category
 3   data_subject_type          22879 non-null  category
 4   data_type                  22879 non-null  category
 5   decision_taken             22879 non-null  category
 6   incident_category          22879 non-null  category
 7   incident_type              22879 non-null  category
 8   no_data_subjects_affected  22879 non-null  category
 9   time_taken_to_report       22879 non-null  category
dtypes: category(10)
memory usage: 764.5 KB

No. data subjects affected values (ordered):
['1 to 9', '10 to 99', '100 to 1k', '1k to 10k', '10k to 100k', '100k and above', 'Unknown']


In [6]:
df_ml = aggregate_to_unique_breaches(df_health_original)
df_ml.info()
print(f"\nDecision taken distribution:\n{df_ml['decision_taken'].value_counts()}")

c:\Users\elsad\OneDrive\Documents\GitHub\data-incident-dashboard\notebooks\..\src\preprocessing.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_aggregated = df_aggregated.groupby("bi_reference").agg({


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9250 entries, 0 to 9249
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   year                       9250 non-null   category
 1   quarter                    9250 non-null   category
 2   data_subject_type          9250 non-null   object  
 3   data_type                  9250 non-null   object  
 4   decision_taken             9250 non-null   category
 5   incident_category          9250 non-null   category
 6   incident_type              9250 non-null   category
 7   no_data_subjects_affected  9250 non-null   category
 8   time_taken_to_report       9250 non-null   category
dtypes: category(7), object(2)
memory usage: 209.9+ KB

Decision taken distribution:
decision_taken
Informal Action Taken    7670
No Further Action         982
Investigation Pursued     598
Not Yet Assigned            0
Name: count, dtype: int64


In [7]:
try:
    df_ml.to_csv(PROCESSED_DATA_PATH, index=False)
    print(f"\nProcessed dataset saved to {PROCESSED_DATA_PATH}")
except Exception as e:
    print(f"\nError saving processed dataset: {e}")


Processed dataset saved to ../data/new-data-security-incident-trends-health-sector.csv


In [8]:
df_processed = preprocess_original_dataset()
validate_processed_dataset(df_processed)

c:\Users\elsad\OneDrive\Documents\GitHub\data-incident-dashboard\notebooks\..\src\preprocessing.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_aggregated = df_aggregated.groupby("bi_reference").agg({
